In [1]:
import pandas as pd
from textblob import TextBlob
from collections import Counter

In [2]:
# Load the dataset
file_path = "Labeled_Olympics_Paris_2024_Tweets.csv"  # Replace with your dataset's path
df = pd.read_csv(file_path)

In [3]:
# Define keywords typically associated with hate speech (expandable list)
hate_keywords = [
    'hate', 'anger', 'disgust', 'shame', 'mock', 'ruin', 'disrespect', 'dishonor',
    'fail', 'lose', 'disappoint', 'weak', 'cheat', 'unfair', 'corrupt', 'scam', 
    'boycott', 'ban', 'protest', 'controversy', 'biased', 'bad', 'terrible', 
    'awful', 'useless', 'overrated', 'waste', 'flop', 'loser', 'choker', 'fraud', 
    'pathetic', 'expensive', 'pollution', 'chaos'
]

In [4]:
# Function to check if a tweet contains hate keywords
def contains_hate_keywords(tweet, keywords):
    if pd.isna(tweet):  # Handle NaN values
        return False
    tweet = tweet.lower()
    return any(keyword in tweet for keyword in keywords)


In [5]:
# Apply the hate keyword check
df['Hate_Keyword_Flag'] = df['Clean_Tweets'].apply(lambda x: contains_hate_keywords(x, hate_keywords))

In [6]:
# Sentiment analysis
def analyze_sentiment(tweet):
    if pd.isna(tweet):  # Handle NaN values
        return "neutral"
    sentiment = TextBlob(tweet).sentiment.polarity
    if sentiment > 0.2:
        return "positive"
    elif sentiment < -0.2:
        return "negative"
    else:
        return "neutral"

In [7]:
df['Sentiment'] = df['Clean_Tweets'].apply(analyze_sentiment)

In [8]:
# Compare labeling with analysis
def check_label(row):
    if row['Label'] == "hate" and row['Sentiment'] == "positive":
        return "Potential Mislabeling"
    if row['Label'] == "non-hate" and row['Hate_Keyword_Flag']:
        return "Potential Mislabeling"
    return "Correct"

In [9]:
df['Label_Validation'] = df.apply(check_label, axis=1)

In [10]:
# Analyze potential mislabelings
mislabelings = df[df['Label_Validation'] == "Potential Mislabeling"]

In [11]:
# Output summary
print("Labeling Summary:")
print(df['Label_Validation'].value_counts())
print("\nPotential Mislabelings:")
print(mislabelings[['Clean_Tweets', 'Label', 'Sentiment', 'Hate_Keyword_Flag']])

Labeling Summary:
Label_Validation
Correct                  29109
Potential Mislabeling      355
Name: count, dtype: int64

Potential Mislabelings:
                                            Clean_Tweets Label Sentiment  \
74     ['letsile', 'tebogo', 'put', 'top', 'world', '...  hate  positive   
157    ['lol', 'var', 'mana', 'timnasday', 'kena', 'a...  hate  positive   
159    ['parisolympics2024', 'ioc', 'member', 'nita',...  hate  positive   
230    ['love', 'arshadnadeem', 'thank', 'pakistanzin...  hate  positive   
242    ['love', 'arshadnadeem', 'thank', 'pakistanzin...  hate  positive   
...                                                  ...   ...       ...   
29305  ['bjp', 'mp', 'hema', 'malini', 'say', 'vinesh...  hate  positive   
29310  ['many', 'time', 'watch', 'understand', 'iocha...  hate  positive   
29360  ['team', 'south', 'men', 'relay', 'final', 'br...  hate  positive   
29373  ['love', 'arshadnadeem', 'thank', 'pakistanzin...  hate  positive   
29403  ['letsile

In [12]:
# Save mislabelings to a CSV file
output_file = "potential_mislabelings.csv"
mislabelings.to_csv(output_file, index=False)
print(f"Potential mislabelings saved to {output_file}")

Potential mislabelings saved to potential_mislabelings.csv
